# Recompute Missing Table Statistics
This notebook reads the raw run data, recomputes variance/median/count values, updates the JSON summary files, and writes insights to a text summary.

In [ ]:
import pandas as pd
from pathlib import Path
REPORT_DIR = Path(r'e:/tasnim/Graduation project model/tasnim/evaluation/statistical_analysis/reports')
SUMMARY_DIR = REPORT_DIR.parent / 'summary'
RAW_CSV = REPORT_DIR.parent / 'raw_data' / 'all_runs.csv'
print('Report dir exists:', REPORT_DIR.exists())
print('Summary dir exists:', SUMMARY_DIR.exists())
print('Raw data exists:', RAW_CSV.exists())

In [ ]:
df = pd.read_csv(RAW_CSV)
metrics = [c for c in df.columns if c not in ['batch_id','run_id']]
print('Metrics:', metrics)
print('Rows:', len(df))
print(df.head())

In [ ]:
def summarize(df):
    out = {}
    for metric in metrics:
        values = df[metric].dropna().astype(float)
        out[metric] = {
            'mean': float(values.mean()),
            'std': float(values.std()),
            'variance': float(values.var()),
            'min': float(values.min()),
            'max': float(values.max()),
            'median': float(values.median()),
            'count': int(values.count()),
        }
    return out
global_stats = summarize(df)
per_batch = {str(batch): summarize(group) for batch, group in df.groupby('batch_id')}
print('Global stats computed for', len(global_stats), 'metrics')
print('Per batch stats computed for', len(per_batch), 'batches')

In [ ]:
import json
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
with open(SUMMARY_DIR / 'global_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(global_stats, f, indent=2)
with open(SUMMARY_DIR / 'per_batch_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(per_batch, f, indent=2)
print('Updated JSON summary files')

In [ ]:
insight = []
max_mean_metric = max(global_stats.items(), key=lambda item: item[1]['mean'])[0]
min_mean_metric = min(global_stats.items(), key=lambda item: item[1]['mean'])[0]
insight.append(f"Highest average metric: {max_mean_metric} with mean {global_stats[max_mean_metric]['mean']:.4f}")
insight.append(f"Lowest average metric: {min_mean_metric} with mean {global_stats[min_mean_metric]['mean']:.4f}")
high_variance = sorted(global_stats.items(), key=lambda item: item[1]['variance'], reverse=True)[:3]
insight.append('Top 3 highest variance metrics: ' + ', '.join([f"{metric} ({stats['variance']:.6f})" for metric, stats in high_variance]))
insight_text = '\n'.join(insight)
print(insight_text)
with open(REPORT_DIR / 'insight_summary.txt', 'w', encoding='utf-8') as f:
    f.write(insight_text)
print('Insight summary saved.')